In [ ]:
!pip install datasets transformers pandas scikit-learn torch

import pandas as pd
import torch
from torch import nn
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
from google.colab import userdata
from huggingface_hub import login

# 1. Authenticate with Hugging Face securely
print("Authenticating with Hugging Face...")
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

print("=== STAGE 1: DOWNLOADING & CONTEXT WINDOWING ===")
# Load dataset
dataset = load_dataset("opennyaiorg/InRhetoricalRoles", split="train")
df = pd.DataFrame(dataset)

extracted_sentences = []
for index, row in df.iterrows():
    annotations = row['annotations'][0]['result']
    for i, anno in enumerate(annotations):
        if 'value' in anno and 'text' in anno['value'] and 'labels' in anno['value']:
            current_text = anno['value']['text'].strip()
            label = anno['value']['labels'][0]

            # The Context Upgrade: Grab the previous sentence
            if i > 0 and 'text' in annotations[i-1]['value']:
                prev_text = annotations[i-1]['value']['text'].strip()
            else:
                prev_text = "START OF DOCUMENT"

            # Combine them using the BERT Separator Token
            combined_text = f"{prev_text} [SEP] {current_text}"

            extracted_sentences.append({
                'sentence': combined_text,
                'label': label
            })

clean_df = pd.DataFrame(extracted_sentences)
print(f"Total contextual sentences extracted: {len(clean_df)}")

# Encode Labels
label_encoder = LabelEncoder()
clean_df['label_id'] = label_encoder.fit_transform(clean_df['label'])
num_labels = len(label_encoder.classes_)
print(f"Found {num_labels} unique rhetorical roles.")

Authenticating with Hugging Face...
=== STAGE 1: DOWNLOADING & CONTEXT WINDOWING ===


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-bb0092e0d85493(…):   0%|          | 0.00/4.87M [00:00<?, ?B/s]

data/dev-00000-of-00001-af55705c75623915(…):   0%|          | 0.00/494k [00:00<?, ?B/s]

data/test-00000-of-00001-2526ab833e27e0e(…):   0%|          | 0.00/736k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/247 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/30 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50 [00:00<?, ? examples/s]

Total contextual sentences extracted: 28986
Found 13 unique rhetorical roles.


In [ ]:
print("=== STAGE 2: TOKENIZATION & DATASET SPLIT ===")
# NEW CODE:
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")

train_texts, val_texts, train_labels, val_labels = train_test_split(
    clean_df['sentence'].tolist(),
    clean_df['label_id'].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=clean_df['label_id'].tolist()
)

class ContextualLegalDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts[index])
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = ContextualLegalDataset(train_texts, train_labels, tokenizer)
val_dataset = ContextualLegalDataset(val_texts, val_labels, tokenizer)
print("Datasets built successfully!")

=== STAGE 2: TOKENIZATION & DATASET SPLIT ===


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Datasets built successfully!


In [ ]:
print("=== STAGE 3: FOCAL LOSS & FREEZING ARCHITECTURE ===")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1_macro': f1}

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Multi-class Focal Loss Math
        ce_loss = nn.CrossEntropyLoss(reduction='none')(logits, labels)
        pt = torch.exp(-ce_loss)
        gamma = 2.0
        focal_loss = ((1 - pt) ** gamma * ce_loss).mean()

        return (focal_loss, outputs) if return_outputs else focal_loss

# Function to generate a fresh, frozen model for the grid search loop
def get_frozen_model():
    # NEW CODE:
    model = AutoModelForSequenceClassification.from_pretrained(
        "allenai/scibert_scivocab_uncased",
        num_labels=num_labels
    )

    # Freeze base embeddings
    for param in model.bert.embeddings.parameters():
        param.requires_grad = False

    # Freeze bottom 6 layers (0 to 5)
    for layer in model.bert.encoder.layer[:6]:
        for param in layer.parameters():
            param.requires_grad = False

    return model

print("Architecture primed and ready for Grid Search.")

=== STAGE 3: FOCAL LOSS & FREEZING ARCHITECTURE ===
Architecture primed and ready for Grid Search.


In [ ]:
import gc

print("=== STAGE 4: HYPERPARAMETER GRID SEARCH ===")

# We test two learning rates. We keep batch size at 16 to avoid Colab memory crashes.
learning_rates = [2e-5, 5e-5]
batch_size = 16

best_f1 = 0
best_lr = None

for lr in learning_rates:
    print(f"\n🚀 Testing Learning Rate: {lr}")

    # 1. Get a fresh, frozen brain
    model = get_frozen_model()

    # 2. Set training rules
    training_args = TrainingArguments(
        output_dir=f"./grid_search_lr{lr}",
        num_train_epochs=2, # Short run just to test the waters
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        eval_strategy="epoch",
        save_strategy="no", # Don't save weights during tests to save disk space
        weight_decay=0.01,
        fp16=torch.cuda.is_available(), # Auto-use mixed precision if on GPU
        report_to="none"
    )

    # 3. Build the custom trainer
    trainer = FocalTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    # 4. Train and Evaluate
    trainer.train()
    metrics = trainer.evaluate()
    current_f1 = metrics['eval_f1_macro']
    print(f"Result for LR {lr}: F1-Macro = {current_f1:.4f}")

    # 5. Track the winner
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_lr = lr

    # 6. Crucial: Clear GPU memory before the next loop to prevent crashes
    del model
    del trainer
    torch.cuda.empty_cache()
    gc.collect()

print("\n=======================================================")
print(f"🏆 GRID SEARCH COMPLETE! WINNING LEARNING RATE: {best_lr}")
print(f"🏆 BEST F1-MACRO SCORE: {best_f1:.4f}")
print("=======================================================")

print("\nNEXT STEP: Create a final TrainingArguments block using LR = " + str(best_lr) + ", set num_train_epochs=5, and save the model!")

=== STAGE 4: HYPERPARAMETER GRID SEARCH ===

🚀 Testing Learning Rate: 2e-05


pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

...

Result for LR 2e-05: F1-Macro = 0.5320

🚀 Testing Learning Rate: 5e-05


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.578279,0.529269,0.682613,0.531481
2,0.365841,0.459287,0.726081,0.577013


Result for LR 5e-05: F1-Macro = 0.5770

🏆 GRID SEARCH COMPLETE! WINNING LEARNING RATE: 5e-05
🏆 BEST F1-MACRO SCORE: 0.5770

NEXT STEP: Create a final TrainingArguments block using LR = 5e-05, set num_train_epochs=5, and save the model!


In [ ]:
import numpy as np
import torch

print("=== STAGE 5: THE FINAL PRODUCTION RUN ===")

# 1. Initialize a fresh, frozen model for the final run
final_model = get_frozen_model()

# 2. Apply the Winning Settings from Stage 4
final_training_args = TrainingArguments(
    output_dir="./legal_expert_brain_v2",
    num_train_epochs=5,                 # Deep learning time
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-05,                # 🏆 THE WINNER
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,        # Automatically keep the absolute best epoch
    metric_for_best_model="f1_macro",   # Judge 'best' by our Focal Loss metric
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# 3. Build the Final Trainer
final_trainer = FocalTrainer(
    model=final_model,
    args=final_training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# 4. Train the Final Model
print("Initiating Final Deep Learning Protocol. This will take about 20 minutes...")
final_trainer.train()

# 5. Save the Output Artifacts for Student 1
print("\nSaving the completed Brain to disk...")
final_trainer.save_model("./legal_expert_brain_v2")
tokenizer.save_pretrained("./legal_expert_brain_v2")

# Save the Answer Key so the web backend knows what the 13 categories are
np.save("./legal_expert_brain_v2/label_classes.npy", label_encoder.classes_)

print("✅ VERSION 2.0 IS COMPLETE AND SAVED!")

=== STAGE 5: THE FINAL PRODUCTION RUN ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

Initiating Final Deep Learning Protocol. This will take about 20 minutes...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.594645,0.551623,0.670883,0.520681
2,0.400293,0.447630,0.716421,0.574987
3,0.236804,0.406763,0.752990,0.643889
4,0.119161,0.441046,0.762649,0.653241
5,0.061850,0.441632,0.775529,0.679358


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Saving the completed Brain to disk...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ VERSION 2.0 IS COMPLETE AND SAVED!


In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Loading Version 3.0 Brain...")

# 1. Load the customized model, tokenizer, and answer key from your saved folder
model_path = "./legal_expert_brain_v2"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval() # Turn off training mode

# Load the answer key
classes = np.load(f"{model_path}/label_classes.npy", allow_pickle=True)

# 2. The Setup (What the website will send to the AI)
previous_sentence = "We must look at how this court has handled similar disputes in the past."
current_sentence = "In the landmark case of Kesavananda Bharati v. State of Kerala, the basic structure doctrine was established."

# Expected Prediction: PRE_RELIED or PRECEDENT

# Combine them exactly how we trained the AI
combined_text = f"{previous_sentence} [SEP] {current_sentence}"
print(f"\nAnalyzing Contextual Input:\n{combined_text}\n")

# 3. Translate to Math
inputs = tokenizer(
    combined_text,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=256
)

# Move inputs to the correct device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# 4. Predict
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

    # Get the highest probability
    predicted_id = torch.argmax(logits, dim=-1).item()
    confidence = torch.softmax(logits, dim=-1)[0][predicted_id].item() * 100

predicted_label = classes[predicted_id]

print("========================================")
print(f"🤖 AI PREDICTION: {predicted_label}")
print(f"📊 CONFIDENCE:    {confidence:.2f}%")
print("========================================")

# ---> ADD THE NEW CODE RIGHT HERE <---
print("\n--- AI's Internal Debate (Top 3 Guesses) ---")
top_3 = torch.topk(torch.softmax(logits, dim=-1)[0], 3)
for i in range(3):
    idx = top_3.indices[i].item()
    score = top_3.values[i].item() * 100
    print(f"Guess {i+1}: {classes[idx]} ({score:.2f}%)")

Loading Version 3.0 Brain...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Analyzing Contextual Input:
We must look at how this court has handled similar disputes in the past. [SEP] In the landmark case of Kesavananda Bharati v. State of Kerala, the basic structure doctrine was established.

🤖 AI PREDICTION: PRE_RELIED
📊 CONFIDENCE:    52.00%

--- AI's Internal Debate (Top 3 Guesses) ---
Guess 1: PRE_RELIED (52.00%)
Guess 2: ANALYSIS (45.89%)
Guess 3: PRE_NOT_RELIED (1.42%)


#using INLEGALBERT

In [ ]:
print("=== STAGE 2: TOKENIZATION & DATASET SPLIT ===")
# NEW CODE:
tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")

train_texts, val_texts, train_labels, val_labels = train_test_split(
    clean_df['sentence'].tolist(),
    clean_df['label_id'].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=clean_df['label_id'].tolist()
)

class ContextualLegalDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts[index])
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = ContextualLegalDataset(train_texts, train_labels, tokenizer)
val_dataset = ContextualLegalDataset(val_texts, val_labels, tokenizer)
print("Datasets built successfully!")

=== STAGE 2: TOKENIZATION & DATASET SPLIT ===


config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Datasets built successfully!


In [ ]:
print("=== STAGE 3: FOCAL LOSS & FREEZING ARCHITECTURE ===")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1_macro': f1}

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Multi-class Focal Loss Math
        ce_loss = nn.CrossEntropyLoss(reduction='none')(logits, labels)
        pt = torch.exp(-ce_loss)
        gamma = 2.0
        focal_loss = ((1 - pt) ** gamma * ce_loss).mean()

        return (focal_loss, outputs) if return_outputs else focal_loss

# Function to generate a fresh, frozen model for the grid search loop
def get_frozen_model():
    # NEW CODE:
    model = AutoModelForSequenceClassification.from_pretrained(
        "law-ai/InLegalBERT",
        num_labels=num_labels
    )

    # Freeze base embeddings
    for param in model.bert.embeddings.parameters():
        param.requires_grad = False

    # Freeze bottom 6 layers (0 to 5)
    for layer in model.bert.encoder.layer[:6]:
        for param in layer.parameters():
            param.requires_grad = False

    return model

print("Architecture primed and ready for Grid Search.")

=== STAGE 3: FOCAL LOSS & FREEZING ARCHITECTURE ===
Architecture primed and ready for Grid Search.


In [ ]:
import gc

print("=== STAGE 4: HYPERPARAMETER GRID SEARCH ===")

# We test two learning rates. We keep batch size at 16 to avoid Colab memory crashes.
learning_rates = [2e-5, 5e-5]
batch_size = 16

best_f1 = 0
best_lr = None

for lr in learning_rates:
    print(f"\n🚀 Testing Learning Rate: {lr}")

    # 1. Get a fresh, frozen brain
    model = get_frozen_model()

    # 2. Set training rules
    training_args = TrainingArguments(
        output_dir=f"./grid_search_lr{lr}",
        num_train_epochs=2, # Short run just to test the waters
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        eval_strategy="epoch",
        save_strategy="no", # Don't save weights during tests to save disk space
        weight_decay=0.01,
        fp16=torch.cuda.is_available(), # Auto-use mixed precision if on GPU
        report_to="none"
    )

    # 3. Build the custom trainer
    trainer = FocalTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    # 4. Train and Evaluate
    trainer.train()
    metrics = trainer.evaluate()
    current_f1 = metrics['eval_f1_macro']
    print(f"Result for LR {lr}: F1-Macro = {current_f1:.4f}")

    # 5. Track the winner
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_lr = lr

    # 6. Crucial: Clear GPU memory before the next loop to prevent crashes
    del model
    del trainer
    torch.cuda.empty_cache()
    gc.collect()

print("\n=======================================================")
print(f"🏆 GRID SEARCH COMPLETE! WINNING LEARNING RATE: {best_lr}")
print(f"🏆 BEST F1-MACRO SCORE: {best_f1:.4f}")
print("=======================================================")

print("\nNEXT STEP: Create a final TrainingArguments block using LR = " + str(best_lr) + ", set num_train_epochs=5, and save the model!")

=== STAGE 4: HYPERPARAMETER GRID SEARCH ===

🚀 Testing Learning Rate: 2e-05


pytorch_model.bin:   0%|          | 0.00/534M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect id

model.safetensors:   0%|          | 0.00/534M [00:00<?, ?B/s]

...

Result for LR 2e-05: F1-Macro = 0.5820

🚀 Testing Learning Rate: 5e-05


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect id

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.488446,0.437279,0.728841,0.602025
2,0.322008,0.371738,0.763569,0.638009


Result for LR 5e-05: F1-Macro = 0.6380

🏆 GRID SEARCH COMPLETE! WINNING LEARNING RATE: 5e-05
🏆 BEST F1-MACRO SCORE: 0.6380

NEXT STEP: Create a final TrainingArguments block using LR = 5e-05, set num_train_epochs=5, and save the model!


In [ ]:
import numpy as np
import torch

print("=== STAGE 5: THE FINAL PRODUCTION RUN ===")

# 1. Initialize a fresh, frozen model for the final run
final_model = get_frozen_model()

# 2. Apply the Winning Settings from Stage 4
final_training_args = TrainingArguments(
    output_dir="./legal_expert_brain_v3",
    num_train_epochs=5,                 # Deep learning time
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-05,                # 🏆 THE WINNER
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,        # Automatically keep the absolute best epoch
    metric_for_best_model="f1_macro",   # Judge 'best' by our Focal Loss metric
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# 3. Build the Final Trainer
final_trainer = FocalTrainer(
    model=final_model,
    args=final_training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# 4. Train the Final Model
print("Initiating Final Deep Learning Protocol. This will take about 20 minutes...")
final_trainer.train()

# 5. Save the Output Artifacts for Student 1
print("\nSaving the completed Brain to disk...")
final_trainer.save_model("./legal_expert_brain_v3")
tokenizer.save_pretrained("./legal_expert_brain_v3")

# Save the Answer Key so the web backend knows what the 13 categories are
np.save("./legal_expert_brain_v3/label_classes.npy", label_encoder.classes_)

print("✅ VERSION 3.0 IS COMPLETE AND SAVED!")

=== STAGE 5: THE FINAL PRODUCTION RUN ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect id

Initiating Final Deep Learning Protocol. This will take about 20 minutes...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.496239,0.443365,0.729531,0.603232
2,0.338128,0.367228,0.750230,0.632021
3,0.210603,0.352617,0.772769,0.671007
4,0.118257,0.345778,0.794848,0.694618
5,0.072457,0.361040,0.801748,0.714078


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Saving the completed Brain to disk...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ VERSION 3.0 IS COMPLETE AND SAVED!


In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Loading Version 3.0 Brain...")

# 1. Load the customized model, tokenizer, and answer key from your saved folder
model_path = "./legal_expert_brain_v3"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval() # Turn off training mode

# Load the answer key
classes = np.load(f"{model_path}/label_classes.npy", allow_pickle=True)

# 2. The Setup (What the website will send to the AI)
previous_sentence = "We must look at how this court has handled similar disputes in the past."
current_sentence = "In the landmark case of Kesavananda Bharati v. State of Kerala, the basic structure doctrine was established."

# Expected Prediction: PRE_RELIED or PRECEDENT

# Combine them exactly how we trained the AI
combined_text = f"{previous_sentence} [SEP] {current_sentence}"
print(f"\nAnalyzing Contextual Input:\n{combined_text}\n")

# 3. Translate to Math
inputs = tokenizer(
    combined_text,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=256
)

# Move inputs to the correct device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# 4. Predict
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

    # Get the highest probability
    predicted_id = torch.argmax(logits, dim=-1).item()
    confidence = torch.softmax(logits, dim=-1)[0][predicted_id].item() * 100

predicted_label = classes[predicted_id]

print("========================================")
print(f"🤖 AI PREDICTION: {predicted_label}")
print(f"📊 CONFIDENCE:    {confidence:.2f}%")
print("========================================")

# ---> ADD THE NEW CODE RIGHT HERE <---
print("\n--- AI's Internal Debate (Top 3 Guesses) ---")
top_3 = torch.topk(torch.softmax(logits, dim=-1)[0], 3)
for i in range(3):
    idx = top_3.indices[i].item()
    score = top_3.values[i].item() * 100
    print(f"Guess {i+1}: {classes[idx]} ({score:.2f}%)")

Loading Version 3.0 Brain...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Analyzing Contextual Input:
We must look at how this court has handled similar disputes in the past. [SEP] In the landmark case of Kesavananda Bharati v. State of Kerala, the basic structure doctrine was established.

🤖 AI PREDICTION: PRE_RELIED
📊 CONFIDENCE:    72.85%

--- AI's Internal Debate (Top 3 Guesses) ---
Guess 1: PRE_RELIED (72.85%)
Guess 2: PRE_NOT_RELIED (19.30%)
Guess 3: ANALYSIS (7.14%)


In [ ]:
import matplotlib.pyplot as plt
from google.colab import files

# 1. Your Exact Final Training Data (InLegalBERT)
epochs = [1, 2, 3, 4, 5]
train_loss = [0.496239, 0.338128, 0.210603, 0.118257, 0.072457]
val_loss = [0.443365, 0.367228, 0.352617, 0.345778, 0.361040]
f1_macro = [0.603232, 0.632021, 0.671007, 0.694618, 0.714078]

# ==========================================
# GRAPH 1: The Loss Curve (Learning Proof)
# ==========================================
plt.figure(figsize=(8, 5))
plt.plot(epochs, train_loss, label='Training Loss', marker='o', color='#3498db', linewidth=2)
plt.plot(epochs, val_loss, label='Validation Loss', marker='o', color='#e74c3c', linewidth=2)

plt.title('InLegalBERT: Training vs. Validation Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.xticks(epochs)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()

# Save and trigger download
plt.savefig('loss_curve.png', dpi=300)
files.download('loss_curve.png')
plt.close()

# ==========================================
# GRAPH 2: The F1-Macro Ascent (Performance Proof)
# ==========================================
plt.figure(figsize=(8, 5))
plt.plot(epochs, f1_macro, label='F1-Macro Score', marker='s', color='#2ecc71', linewidth=2)

# Annotate the final peak score
plt.annotate(f'Peak: {f1_macro[-1]*100:.2f}%',
             xy=(5, f1_macro[-1]),
             xytext=(4.2, f1_macro[-1] - 0.02),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6),
             fontsize=11, fontweight='bold')

plt.title('InLegalBERT: F1-Macro Progression Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('F1-Macro Score', fontsize=12)
plt.xticks(epochs)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()

# Save and trigger download
plt.savefig('f1_macro_curve.png', dpi=300)
files.download('f1_macro_curve.png')
plt.close()

print("✅ Graphs successfully generated and downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Graphs successfully generated and downloaded!


In [ ]:
import matplotlib.pyplot as plt
from google.colab import files

# The Epochs
epochs = [1, 2, 3, 4, 5]

# The Exact F1-Macro Data from your 3 training runs
f1_scibert = [0.520681, 0.574987, 0.643889, 0.653241, 0.679358]
f1_legalbert = [0.574102, 0.601492, 0.672886, 0.692118, 0.703779]
f1_inlegalbert = [0.603232, 0.632021, 0.671007, 0.694618, 0.714078]

# ==========================================
# The Ultimate Comparative Graph
# ==========================================
plt.figure(figsize=(9, 6))

# Plot all three lines
plt.plot(epochs, f1_scibert, label='SciBERT (Academic Base)', marker='^', color='#e74c3c', linewidth=2.5, linestyle='--')
plt.plot(epochs, f1_legalbert, label='Legal-BERT (Global Law)', marker='s', color='#f39c12', linewidth=2.5, linestyle='--')
plt.plot(epochs, f1_inlegalbert, label='InLegalBERT (Indian Law - CHAMPION)', marker='o', color='#2ecc71', linewidth=3.5)

# Annotate the final peak scores
plt.annotate('67.9%', xy=(5, f1_scibert[-1]), xytext=(5.1, f1_scibert[-1] - 0.005), fontweight='bold', color='#e74c3c')
plt.annotate('70.3%', xy=(5, f1_legalbert[-1]), xytext=(5.1, f1_legalbert[-1] - 0.005), fontweight='bold', color='#f39c12')
plt.annotate('71.4%', xy=(5, f1_inlegalbert[-1]), xytext=(5.1, f1_inlegalbert[-1] - 0.005), fontweight='bold', color='#27ae60')

plt.title('Ablation Study: F1-Macro Progression Across Models', fontsize=15, fontweight='bold')
plt.xlabel('Training Epochs', fontsize=12)
plt.ylabel('F1-Macro Score', fontsize=12)
plt.xticks(epochs)
plt.xlim(0.8, 5.5) # Give a little extra room on the right for the text
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()

# Save and trigger download
plt.savefig('comparative_f1_study.png', dpi=300)
files.download('comparative_f1_study.png')
plt.close()

print("✅ Ultimate Comparative Graph generated and downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Ultimate Comparative Graph generated and downloaded!
